# LoRA fine-tune LARGE SigLIP-2 for PAR (Colab T4)

Files live in Drive at `MyDrive/PAR_Project/` (`train_lora.py` + `PA100K.zip`).
Select Kernel -> Colab -> **T4 GPU** -> run cells top to bottom.

In [ ]:
# 1. Confirm T4 GPU
!nvidia-smi -L

In [ ]:
# 2. Install deps (torch already on Colab)
!pip install -q transformers peft sentencepiece scipy

In [ ]:
# 3. Mount Drive, copy script, unzip data to fast local disk
from google.colab import drive
drive.mount('/content/drive')
import shutil, os
os.makedirs('/content/work/features', exist_ok=True)
shutil.copy('/content/drive/MyDrive/PAR_Project/train_lora.py', '/content/work/train_lora.py')
!cp '/content/drive/MyDrive/PAR_Project/PA100K.zip' /content/PA100K.zip
!unzip -q /content/PA100K.zip -d /content/data
%cd /content/work
!find /content/data -maxdepth 3 -type d

In [ ]:
# 4. Set dataset paths (adjust if the tree above looks different)
ANN = '/content/data/PA-100K/annotation/annotation.mat'
IMG = '/content/data/PA-100K/data/release_data/release_data'
import os; assert os.path.exists(ANN) and os.path.isdir(IMG), 'check paths vs the tree above'
print('OK')

In [ ]:
# 5. Smoke test: 2000 imgs, 1 epoch (~5 min) -- confirms it runs
!python train_lora.py --ann "$ANN" --img_dir "$IMG" --epochs 1 --batch 32 --limit_train 2000 --out /content/work/features

In [ ]:
# 6. Full LoRA fine-tune (all 80k, 6 epochs). A few hours on T4.
#    If out-of-memory, lower --batch to 16.
!python train_lora.py --ann "$ANN" --img_dir "$IMG" --epochs 6 --batch 32 --out /content/work/features

In [ ]:
# 7. Save trained LoRA weights back to Drive
!mkdir -p '/content/drive/MyDrive/PAR_Project/features'
!cp /content/work/features/lora_par.pt '/content/drive/MyDrive/PAR_Project/features/'
print('saved lora_par.pt to Drive/PAR_Project/features')